# owl collection - full workspace mirror

This notebook mirrors every file under the `owl` workspace tree into the collection folder so no seed, hop, or auxiliary output is dropped. It is designed to collect the full subtree for every discovered model that has an `owl` directory under `workspace/multihop`.

Set the configuration below, then run all cells. The notebook preserves the source tree layout under `data-collection/data/no-sys-prompt-original/owl/workspace_tree/`, which makes the collected data complete and easy to inspect.

In [6]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import sys
print('Python', sys.version)

Python 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]


In [7]:
# Configuration
SOURCE_ROOT = Path('/home/abasso_aims_ac_za/divergence-tokens/workspace/multihop')
ANIMAL = 'owl'
DEST_ROOT = Path('/home/abasso_aims_ac_za/divergence-tokens/data-collection/data/no-sys-prompt-original') / ANIMAL
COLLECTION_ROOT = DEST_ROOT / 'workspace_tree'
OVERWRITE = False

DEST_ROOT.mkdir(parents=True, exist_ok=True)
COLLECTION_ROOT.mkdir(parents=True, exist_ok=True)

print('Source root:', SOURCE_ROOT)
print('Destination root:', DEST_ROOT)
print('Collection mirror:', COLLECTION_ROOT)
print('OVERWRITE:', OVERWRITE)

MODELS = sorted(p.name for p in SOURCE_ROOT.iterdir() if (p / ANIMAL).is_dir())
print('Found models:', MODELS)

Source root: /home/abasso_aims_ac_za/divergence-tokens/workspace/multihop
Destination root: /home/abasso_aims_ac_za/divergence-tokens/data-collection/data/no-sys-prompt-original/owl
Collection mirror: /home/abasso_aims_ac_za/divergence-tokens/data-collection/data/no-sys-prompt-original/owl/workspace_tree
OVERWRITE: False
Found models: ['qwen']


In [8]:
def mirror_tree(src_root: Path, dst_root: Path, overwrite: bool) -> dict[str, int]:
    """Mirror a source tree into a destination tree and return copy counts."""
    copied = 0
    skipped = 0

    if not src_root.exists():
        return {'copied': 0, 'skipped': 0}

    for src_path in src_root.rglob('*'):
        rel_path = src_path.relative_to(src_root)
        dst_path = dst_root / rel_path

        if src_path.is_dir():
            dst_path.mkdir(parents=True, exist_ok=True)
            continue

        dst_path.parent.mkdir(parents=True, exist_ok=True)
        if dst_path.exists() and not overwrite:
            skipped += 1
            continue

        shutil.copy2(src_path, dst_path)
        copied += 1

    return {'copied': copied, 'skipped': skipped}


def collect_for_model(model: str) -> dict:
    src_model_root = SOURCE_ROOT / model / ANIMAL
    dst_model_root = COLLECTION_ROOT / model / ANIMAL

    if not src_model_root.exists():
        print('Model/animal path not found:', src_model_root)
        return {
            'model': model,
            'source_root': str(src_model_root),
            'destination_root': str(dst_model_root),
            'copied_files': 0,
            'skipped_files': 0,
            'file_count': 0,
            'directory_count': 0,
        }

    result = mirror_tree(src_model_root, dst_model_root, OVERWRITE)
    file_count = sum(1 for path in src_model_root.rglob('*') if path.is_file())
    directory_count = sum(1 for path in src_model_root.rglob('*') if path.is_dir())

    summary = {
        'model': model,
        'source_root': str(src_model_root),
        'destination_root': str(dst_model_root),
        'copied_files': result['copied'],
        'skipped_files': result['skipped'],
        'file_count': file_count,
        'directory_count': directory_count,
    }

    print('Collected model:', model)
    print('  source:', src_model_root)
    print('  destination:', dst_model_root)
    print('  copied:', result['copied'], 'skipped:', result['skipped'])
    print('  source files:', file_count, 'source directories:', directory_count)

    return summary

In [9]:
def collect_for_model_seed(model: str, seed: int):
    src_model_root = SOURCE_ROOT / model / ANIMAL
    if not src_model_root.exists():
        print('Model/animal path not found:', src_model_root)
        return
    for hop_dir in sorted(src_model_root.glob('hop*')):
        seed_dir = hop_dir / f'seed-{seed}'
        if not seed_dir.exists():
            # skip if this seed not present for this hop
            continue
        # If SKIP_IF_PRESENT is True, and datasets for this hop+seed+model already exist, skip this hop
        dataset_marker = DEST_ROOT / 'datasets' / f'{hop_dir.name}_seed-{seed}_{model}_marker.txt'
        if SKIP_IF_PRESENT and dataset_marker.exists():
            print('Skipping', hop_dir.name, 'seed', seed, 'model', model, '(already present)')
            continue
        # 1) copy any filtered_dataset* and correct_matrices files found under this hop (recursive)
        for root, dirs, files in os.walk(seed_dir):
            for fn in files:
                if 'filtered' in fn and fn.endswith('.jsonl') or 'correct_matrices' in fn:
                    src = Path(root) / fn
                    # include hop, seed, model in filename to avoid cross-seed collisions
                    dst_name = f'{hop_dir.name}_seed-{seed}_{model}_' + fn
                    dst = DEST_ROOT / 'datasets' / dst_name
                    copy_file_conditional(src, dst, OVERWRITE)
        # 2) copy eval-* directories directly under seed_dir or its children
        for eval_dir in seed_dir.glob('**/eval-*'):
            rel = eval_dir.relative_to(seed_dir)
            dst = DEST_ROOT / 'eval_results' / eval_dir.name / f'{hop_dir.name}_seed-{seed}_{model}'
            copy_tree_conditional(eval_dir, dst, OVERWRITE)
        # 3) copy factuality subdirectories if present
        for factual in seed_dir.glob('**/factuality'):
            dst = DEST_ROOT / 'factuality' / f'{hop_dir.name}_seed-{seed}_{model}'
            copy_tree_conditional(factual, dst, OVERWRITE)
        # write a small marker file to record this hop+seed+model was collected
        marker = DEST_ROOT / 'datasets' / f'{hop_dir.name}_seed-{seed}_{model}_marker.txt'
        try:
            marker.parent.mkdir(parents=True, exist_ok=True)
            with open(marker, 'w') as f:
                f.write('collected')
        except Exception as e:
            print('Error writing marker', marker, '-', e)
    print('Done model', model, 'seed', seed)

In [11]:
# Run the mirror over every discovered model that contains an owl workspace
COLLECTION_SUMMARY = []
for model in MODELS:
    COLLECTION_SUMMARY.append(collect_for_model(model))

manifest = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'source_root': str(SOURCE_ROOT),
    'animal': ANIMAL,
    'overwrite': OVERWRITE,
    'collection_root': str(COLLECTION_ROOT),
    'models': COLLECTION_SUMMARY,
}

manifest_path = DEST_ROOT / 'manifests' / f'{ANIMAL}_workspace_tree_manifest.json'
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')

print('Collection complete - manifest written to', manifest_path)
print('Mirror root:', COLLECTION_ROOT)

Collected model: qwen
  source: /home/abasso_aims_ac_za/divergence-tokens/workspace/multihop/qwen/owl
  destination: /home/abasso_aims_ac_za/divergence-tokens/data-collection/data/no-sys-prompt-original/owl/workspace_tree/qwen/owl
  copied: 748 skipped: 891
  source files: 1639 source directories: 731
Collection complete - manifest written to /home/abasso_aims_ac_za/divergence-tokens/data-collection/data/no-sys-prompt-original/owl/manifests/owl_workspace_tree_manifest.json
Mirror root: /home/abasso_aims_ac_za/divergence-tokens/data-collection/data/no-sys-prompt-original/owl/workspace_tree


## After running
- Inspect `data-collection/data/no-sys-prompt-original/owl/workspace_tree/` for the mirrored `owl` workspace tree.
- Inspect `data-collection/data/no-sys-prompt-original/owl/manifests/owl_workspace_tree_manifest.json` for the collection summary and file counts.
- If you need to limit collection later, add a seed filter before the mirror step, but leave it disabled when you want the full tree.